In [14]:
import os
import shutil
from MRE import Job # Importamos el emulador

# --- FASE MAP ---
# Se ejecuta por cada línea. key = ID local, value = "ID_visitante ap_local ap_vis"
# Hay un map definido por dataset para etiquetar el origen (EST / PREM).

# Filtramos el dataset Estandar
def fmap_est(key, value, context):
    datos = value.split() # separa la línea en campos
    id_local = key  # ID_equipo_local es la clave
    id_vis = datos[0]  # ID_equipo_visitante
    ap_local = int(datos[1])  # usuarios que apostaron al local como ganador
    ap_vis = int(datos[2])  # usuarios que apostaron al visitante como ganador

    # Emitimos dos tuplas intemedias por equipo (local y visitante)
    context.write(id_local, ("EST", ap_local))
    context.write(id_vis, ("EST", ap_vis))

# Filtramos el dataset Premium
def fmap_prem(key, value, context):
    datos = value.split() # separa la línea en campos
    id_local = key  # ID_equipo_local es la clave
    id_vis = datos[0]  # ID_equipo_visitante
    ap_local = int(datos[1])  # usuarios que apostaron al local como ganador
    ap_vis = int(datos[2])  # usuarios que apostaron al visitante como ganador

    # Emitimos dos tuplas intemedias por equipo (local y visitante)
    context.write(id_local, ("PREM", ap_local))
    context.write(id_vis, ("PREM", ap_vis))

# --- FASE COMBINER ---
# Suma parcial por equipo y origen, solo con lo que vio su mapper.
# Reduce el tráfico de red. NO filtra por X (ya que solo ve datos parciales).
def fcomb(key, values, context):
    total_est = 0
    total_prem = 0

    for v in values:
        origen = v[0]
        apuestas = v[1]
        if origen == "EST":
            total_est += apuestas # Sumamos las apuestas estandar
        elif origen == "PREM":
            total_prem += apuestas # Suamamos las apuestas premium

    # Emitimos los subtotales, si son mayor a cero para no enviar ceros que no sirvan
    if total_est > 0:
        context.write(key, ("EST", total_est))
    if total_prem > 0:
        context.write(key, ("PREM", total_prem))

# --- FASE REDUCE ---
# Recibe todos los valores de UN equipo (key) ya agrupados por el shuffle.
# Acá se calculan los totales definitivos y se aplica el filtro de X.
def fred(key, values, context):
    total_est = 0
    total_prem = 0

    for v in values:
        origen = v[0]
        apuestas = v[1]
        if origen == "EST":
            total_est += apuestas # Sumamos los subtotales en un total final para estandar
        elif origen == "PREM":
            total_prem += apuestas # Sumamos los subtotales en un total final para premium

    if total_est > context["X"] or total_prem > context["X"]: # Si el total supera X cantidad lo imprimimos
        context.write(key, "")

# --- DRIVER ---
# Configura y dispara el job
if __name__ == '__main__':
    input_est = "/content/in_estandar"     # carpeta con el dataset estándar
    input_prem = "/content/in_premium"     # carpeta con el dataset premium
    output_dir = "/content/salida_punto1"  # carpeta de resultados

    # Preparación: crear carpetas de entrada y copiar los archivos
    os.makedirs(input_est, exist_ok=True)
    os.makedirs(input_prem, exist_ok=True)
    shutil.copy("/content/apuestasEstandar.txt", input_est + "/apuestasEstandar.txt")
    shutil.copy("/content/apuestasPremium.txt", input_prem + "/apuestasPremium.txt")

    # Borrar la salida anterior (si existe) para que no falle al re-ejecutar
    shutil.rmtree(output_dir, ignore_errors=True)

    job1 = Job(input_est, output_dir, fmap_est, fred) # Job con el primer dataset estándar y su map correspondiente
    job1.addInputPath(input_prem, fmap_prem) # Segundo dataset premium con su map
    job1.setCombiner(fcomb) # Combiner para reducir datos enviados por la red
    job1.setParams({"X": 1000}) # Parametro X ajustable a la consulta

    success = job1.waitForCompletion()

    if success:
        print("El job finalizó correctamente!")

El job finalizó correctamente!
